# Experiment 3 · Few-shot LoRA vs traditional baselines

Foundation models (SAM2, MedSAM, MedSAM-2, SAM3) are **LoRA-fine-tuned**
(rank `r=16`, `alpha=32`, targeting the ViT attention `qkv`/`proj` projections)
on increasing fractions of each dataset's training split —
`0.05 / 0.1 / 0.25 / 0.5` — and evaluated with oracle box prompts.
Traditional CNN baselines (U-Net, TransUNet) are trained on the **same
fractions** so the two families are compared point-for-point along the
data-efficiency curve. Every run uses the fixed global seed `42` and
patient-level splits. This experiment produces the data-efficiency curves of
Figure 1 and the few-shot columns of Table 3.

> Note: SAM (ViT-H) is not LoRA-adaptable and is excluded here. The $f{=}100\%$ foundation point (full supervision) is produced by the Experiment 1 foundation notebook, not this one.

In [ ]:
import os

# Move to the repository root so `thyroidbench`, `data/`, and the experiment
# scripts all resolve from the same working directory.
while not os.path.isdir("thyroidbench"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        raise RuntimeError("Could not locate repository root (no 'thyroidbench/' found).")
    os.chdir(parent)

print("Working directory:", os.getcwd())

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


### Prerequisites

- The foundation checkpoints must be present under `pretrained_models/`
  (e.g. `sam2.1_hiera_large.pt`, `medsam_vit_b.pth`, `MedSAM2_latest.pt`,
  the SAM3 weights, and the SAM ViT-H checkpoint). The LoRA wrapper loads the
  frozen backbone from there.
- The processed datasets and split CSVs must be available under `data/`
  (`data/processed/<dataset>/` and `data/splits/`).
- Training logs to Weights & Biases (project `thyroidbench`); set
  `WANDB_MODE=offline` if you are running without a wandb account.

## Run — foundation LoRA

In [ ]:
for model in ['sam2', 'medsam', 'medsam2', 'sam3']:
    for dataset in ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']:
        for frac in [0.05, 0.1, 0.25, 0.5]:
            !python experiments/exp3_fewshot_lora/run.py --model {model} --dataset {dataset} --fraction {frac}

## Run — traditional CNN baselines at fractions

In [ ]:
for model in ['unet', 'transunet']:
    for dataset in ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']:
        for frac in [0.05, 0.1, 0.25, 0.5]:
            !python experiments/exp3_fewshot_lora/run_traditional.py --model {model} --dataset {dataset} --fraction {frac}

## Results

The per-cell means (DSC / IoU / HD95 for every model x dataset x fraction) are
collated by `aggregate_fewshot.py` into `results/stats/exp3_summary.csv`.

In [ ]:
import pandas as pd

table = pd.read_csv(
    "experiments/exp3_fewshot_lora/results/stats/exp3_summary.csv"
)
table